---

Cloud test

---

In [11]:
import os
import json
import getpass
from google.cloud import storage
from google.cloud.sql.connector import Connector, IPTypes
from google.oauth2.service_account import Credentials
import sqlalchemy

# ==========================================
# 1. DATABASE & CLOUD CONFIGURATION
# ==========================================
INSTANCE_CONNECTION_NAME = "hs-meg-opm:us-central1:neuro-db"
DB_USER = "postgres"
DB_NAME = "postgres"               
BUCKET_NAME = "hs-meg-opm-raw-data" 

# Secure password prompt & JSON Key configuration
DB_PASS = getpass.getpass(prompt="Enter PostgreSQL Password for user 'postgres': ")
KEY_PATH = "/Users/justin/data/cloud/hs-meg-opm-key.json" 

# Load Credentials for both GCS and Cloud SQL
credentials = Credentials.from_service_account_file(
    KEY_PATH, 
    scopes=["https://www.googleapis.com/auth/sqlservice.admin", "https://www.googleapis.com/auth/devstorage.read_write"]
)

# Initialize GCS Client and Cloud SQL Connector
storage_client = storage.Client(credentials=credentials)
connector = Connector(credentials=credentials)

# Initialize SQLAlchemy Connection Pool
def get_db_pool() -> sqlalchemy.engine.Engine:
    getconn = lambda: connector.connect(
        INSTANCE_CONNECTION_NAME, "pg8000",
        user=DB_USER, password=DB_PASS, db=DB_NAME, ip_type=IPTypes.PUBLIC
    )
    return sqlalchemy.create_engine("postgresql+pg8000://", creator=getconn)

db_engine = get_db_pool()


# ==========================================
# 2. UNIFIED PUSH FUNCTION
# ==========================================
def pglPushCloud(experiment_record, local_datafile_path):
    """
    Pushes an experiment record to Cloud SQL AND uploads the datafile to GCS.
    
    :param experiment_record: dict containing keys: "subjectID", "projectName", "sessionID", "experimentSettings"
    :param local_datafile_path: string path of the raw file on your laptop (e.g., "./sub-01_raw.fif")
    """
    subject_id = experiment_record["subjectID"]
    session_id = experiment_record["sessionID"]
    project_name = experiment_record["projectName"]
    
    # 1. Determine GCS upload path dynamically based on metadata
    file_extension = os.path.splitext(local_datafile_path)[1]
    gcs_blob_name = f"subjects/{subject_id}/{session_id}/raw_data{file_extension}"
    
    # 2. Upload the file to Google Cloud Storage
    if not os.path.isfile(local_datafile_path):
        raise FileNotFoundError(f"Local file not found: {local_datafile_path}")
        
    print(f"\n[GCS] Uploading '{local_datafile_path}' to gs://{BUCKET_NAME}/{gcs_blob_name}...")
    bucket = storage_client.bucket(BUCKET_NAME)
    blob = bucket.blob(gcs_blob_name)
    blob.upload_from_filename(local_datafile_path)
    
    # 3. Embed the file's GCS URI directly inside the experimentSettings metadata
    gcs_uri = f"gs://{BUCKET_NAME}/{gcs_blob_name}"
    settings = experiment_record.get("experimentSettings", {})
    settings["gcs_file_uri"] = gcs_uri
    
    # 4. Insert or update the record in the Postgres Database
    query = sqlalchemy.text("""
        INSERT INTO "megTest" ("subjectID", "projectName", "sessionID", "experimentSettings")
        VALUES (:subjectID, :projectName, :sessionID, :experimentSettings)
        ON CONFLICT ("subjectID", "sessionID") 
        DO UPDATE SET 
            "projectName" = EXCLUDED."projectName",
            "experimentSettings" = EXCLUDED."experimentSettings",
            "datetime" = CURRENT_TIMESTAMP;
    """)
    
    try:
        with db_engine.connect() as conn:
            with conn.begin():
                conn.execute(query, {
                    "subjectID": subject_id,
                    "projectName": project_name,
                    "sessionID": session_id,
                    "experimentSettings": json.dumps(settings)
                })
        print(f"[SQL] Successfully pushed record for {subject_id} ({session_id}) to database!")
    except Exception as e:
        print(f"[SQL] Error inserting metadata: {e}")


# ==========================================
# 3. UNIFIED PULL FUNCTION
# ==========================================
def pglPullCloud(subject_id, session_id, local_destination_path):
    """
    Pulls the metadata record from Cloud SQL and downloads the corresponding GCS datafile.
    
    :param subject_id: string ID of the subject (e.g., "sub-101")
    :param session_id: string ID of the session (e.g., "sess-alpha")
    :param local_destination_path: Path on your laptop where the datafile will be downloaded
    :return: dict (the experiment record from the database)
    """
    # 1. Fetch metadata from Cloud SQL
    query = sqlalchemy.text("""
        SELECT "subjectID", "projectName", "sessionID", "datetime", "experimentSettings" 
        FROM "megTest"
        WHERE "subjectID" = :subject_id AND "sessionID" = :session_id;
    """)
    
    record = None
    try:
        with db_engine.connect() as conn:
            result = conn.execute(query, {"subject_id": subject_id, "session_id": session_id})
            row = result.fetchone()
            if row:
                record = dict(zip(result.keys(), row))
            else:
                print(f"[SQL] No record found for Subject: {subject_id}, Session: {session_id}")
                return None
    except Exception as e:
        print(f"[SQL] Error pulling metadata: {e}")
        return None

    # 2. Extract GCS File URI from settings
    settings = record.get("experimentSettings", {})
    gcs_uri = settings.get("gcs_file_uri")
    
    if not gcs_uri:
        print("[GCS] Warning: No file URI associated with this record in database.")
        return record
        
    # Parse the bucket and blob name out of "gs://bucket-name/path/to/blob"
    path_without_scheme = gcs_uri.replace("gs://", "")
    bucket_name, blob_name = path_without_scheme.split("/", 1)
    
    # 3. Download the binary file from GCS
    try:
        os.makedirs(os.path.dirname(os.path.abspath(local_destination_path)), exist_ok=True)
        print(f"\n[GCS] Downloading {gcs_uri} to '{local_destination_path}'...")
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        blob.download_to_filename(local_destination_path)
        print("[GCS] File download completed successfully!")
    except Exception as e:
        print(f"[GCS] Error downloading file: {e}")
        
    return record


# ==========================================
# 4. TESTING THE INTEGRATED WORKFLOW
# ==========================================
if __name__ == "__main__":
    # Create mock dataset file locally
    my_local_file = "/Users/justin/data/cloud/test.fif"
    with open(my_local_file, "wb") as f:
        f.write(b"RAW_MEG_MAGNETOMETER_VALUES_123456789")

    # Define metadata (Notice NO file paths are manually defined here)
    metadata = {
        "subjectID": "sub-102",
        "projectName": "VisualAttention",
        "sessionID": "sess-beta",
        "experimentSettings": {
            "sampleRateHz": 1200,
            "systemModel": "Elekta Neuromag"
        }
    }

    # Test the Push
    pglPushCloud(metadata, my_local_file)

    # Test the Pull (to a different path on your laptop)
    downloaded_file_path = "/Users/justin/data/cloud/sub-102_sess-beta_sensor_readings.fif"
    retrieved_metadata = pglPullCloud(
        subject_id="sub-102", 
        session_id="sess-beta", 
        local_destination_path=downloaded_file_path
    )
    
    # Print the returned experiment record
    print("\nRetrieved Metadata from Cloud SQL:")
    print(json.dumps(retrieved_metadata, indent=2, default=str))
    
    # Clean up connection
    connector.close()



[GCS] Uploading '/Users/justin/data/cloud/test.fif' to gs://hs-meg-opm-raw-data/subjects/sub-102/sess-beta/raw_data.fif...
[SQL] Successfully pushed record for sub-102 (sess-beta) to database!

[GCS] Downloading gs://hs-meg-opm-raw-data/subjects/sub-102/sess-beta/raw_data.fif to '/Users/justin/data/cloud/sub-102_sess-beta_sensor_readings.fif'...
[GCS] File download completed successfully!

Retrieved Metadata from Cloud SQL:
{
  "subjectID": "sub-102",
  "projectName": "VisualAttention",
  "sessionID": "sess-beta",
  "datetime": "2026-09-24 23:35:34.405653+00:00",
  "experimentSettings": {
    "systemModel": "Elekta Neuromag",
    "gcs_file_uri": "gs://hs-meg-opm-raw-data/subjects/sub-102/sess-beta/raw_data.fif",
    "sampleRateHz": 1200
  }
}


In [12]:
def pglPrintAllRecords():
    """Queries and prints all records from the 'megTest' table."""
    query = sqlalchemy.text('SELECT * FROM "megTest";')
    
    try:
        with db_engine.connect() as conn:
            result = conn.execute(query)
            keys = result.keys()
            rows = result.fetchall()
            
            print(f"\n=== Found {len(rows)} Records in 'megTest' ===")
            for row in rows:
                record = dict(zip(keys, row))
                print(json.dumps(record, indent=2, default=str))
                print("-" * 40)
    except Exception as e:
        print(f"Error querying database: {e}")


In [13]:
pglPrintAllRecords()


=== Found 2 Records in 'megTest' ===
{
  "subjectID": "sub-999",
  "projectName": "MEG-Sandbox",
  "sessionID": "sess-test01",
  "datetime": "2026-09-24 23:26:06.570809+00:00",
  "experimentSettings": {
    "sampleRateHz": 1200,
    "sensorsActive": [
      "MEG",
      "EEG"
    ],
    "triggerDelayMs": 4.5
  }
}
----------------------------------------
{
  "subjectID": "sub-102",
  "projectName": "VisualAttention",
  "sessionID": "sess-beta",
  "datetime": "2026-09-24 23:35:34.405653+00:00",
  "experimentSettings": {
    "systemModel": "Elekta Neuromag",
    "gcs_file_uri": "gs://hs-meg-opm-raw-data/subjects/sub-102/sess-beta/raw_data.fif",
    "sampleRateHz": 1200
  }
}
----------------------------------------
